In [ ]:
library(ArchR)
library(dplyr)
library(tibble)
library(SummarizedExperiment)
library(GenomicRanges)
library(Seurat)
library(S4Vectors)


# ------------------------------------------------------------
# 0) Load the ArchR project that already contains RNA labels
# ------------------------------------------------------------
archr_proj <- loadArchRProject(
  "/path/to/nature_study/ArchR/ArchRProject_finalRNA"
)

# sanity checks
table(archr_proj$rna_celltype, useNA = "ifany")
table(archr_proj$stem_core, useNA = "ifany")
table(archr_proj$condition, useNA = "ifany")
table(archr_proj$donor, useNA = "ifany")

In [ ]:
# ------------------------------------------------------------
# 1) Keep the informative hematopoietic groups for ATAC validation
#    (drop unassigned / contaminant-like groups)
# ------------------------------------------------------------
keep_labels <- c(
  "HSC_MPP",
  "B_primed_prog",
  "erythroid_prog",
  "erythroid_heme_prog"
)

meta0 <- as.data.frame(getCellColData(
  archr_proj,
  select = c("rna_celltype", "stem_core", "condition", "donor", "Sample")
)) %>%
  rownames_to_column("cellNames")

cells_keep <- meta0 %>%
  filter(
    !is.na(rna_celltype),
    rna_celltype %in% keep_labels,
    !is.na(donor),
    !is.na(condition)
  ) %>%
  pull(cellNames)

length(cells_keep)

archr_heme <- subsetArchRProject(
  ArchRProj = archr_proj,
  cells = cells_keep,
  outputDirectory = "/path/to/nature_study/ArchR/ArchRProject_ATACvalidate",
  force = TRUE
)

table(archr_heme$rna_celltype, useNA = "ifany")
table(archr_heme$condition, useNA = "ifany")

In [ ]:
# ------------------------------------------------------------
# 2) Add combined grouping columns for condition- and donor-aware ATAC work
# ------------------------------------------------------------
meta <- as.data.frame(getCellColData(
  archr_heme,
  select = c("rna_celltype", "stem_core", "condition", "donor", "Sample")
)) %>%
  rownames_to_column("cellNames") %>%
  mutate(
    rna_celltype_condition = paste(rna_celltype, condition, sep = "__"),
    rna_celltype_donor     = paste(rna_celltype, donor, sep = "__"),
    stem_condition         = ifelse(stem_core == "stem_core",
                                    paste("stem_core", condition, sep = "__"),
                                    NA),
    stem_donor             = ifelse(stem_core == "stem_core",
                                    paste("stem_core", donor, sep = "__"),
                                    NA)
  )

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$rna_celltype_condition,
  name = "rna_celltype_condition",
  cells = meta$cellNames,
  force = TRUE
)

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$rna_celltype_donor,
  name = "rna_celltype_donor",
  cells = meta$cellNames,
  force = TRUE
)

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$stem_condition,
  name = "stem_condition",
  cells = meta$cellNames,
  force = TRUE
)

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$stem_donor,
  name = "stem_donor",
  cells = meta$cellNames,
  force = TRUE
)

table(archr_heme$rna_celltype_condition, useNA = "ifany")
table(archr_heme$rna_celltype_donor, useNA = "ifany")

In [ ]:
addArchRGenome("hg38", install = TRUE)
library(BSgenome.Hsapiens.UCSC.hg38)

In [ ]:
# ------------------------------------------------------------
# 3) Donor-aware group coverages by RNA-defined cell type
#    This is the coverage layer used for peak calling / browser tracks
# ------------------------------------------------------------
archr_heme <- addGroupCoverages(
  ArchRProj = archr_heme,
  groupBy = "rna_celltype",
  useLabels = TRUE,
  sampleLabels = "donor",
  minCells = 40,
  maxCells = 500,
  minReplicates = 2,
  maxReplicates = 5,
  force = TRUE
)

In [ ]:
pathToMacs2 <- "/data/leuven/345/vsc34531/miniconda/envs/macs2_archr_py310/bin/macs2"

file.exists(pathToMacs2)
system2(pathToMacs2, "--version", stdout = TRUE, stderr = TRUE)

In [ ]:
# ------------------------------------------------------------
# 4) Call a union reproducible peak set from the RNA-defined cell types
# ------------------------------------------------------------
archr_heme <- addReproduciblePeakSet(
  ArchRProj = archr_heme,
  groupBy = "rna_celltype",
  peakMethod = "Macs2",
  pathToMacs2 = pathToMacs2,
  reproducibility = "2",
  minCells = 25,
  peaksPerCell = 500,
  force = TRUE
)

In [ ]:
# ------------------------------------------------------------
# 5) Build the PeakMatrix on that peak set
# ------------------------------------------------------------
archr_heme <- addPeakMatrix(
  ArchRProj = archr_heme,
  force = TRUE
)

getAvailableMatrices(archr_heme)

In [ ]:
# ------------------------------------------------------------
# 6) Donor-level grouped ATAC matrix (one column per celltype x donor)
# ------------------------------------------------------------
se_peak_donor <- getGroupSE(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_donor",
  divideN = FALSE,
  scaleTo = NULL
)

se_peak_donor

In [ ]:
# ------------------------------------------------------------
# 7) Build donor-level metadata for the grouped ATAC matrix
# ------------------------------------------------------------
peak_coldata <- as.data.frame(colData(se_peak_donor)) %>%
  rownames_to_column("group_id") %>%
  mutate(
    rna_celltype = sub("__.*$", "", group_id),
    donor = sub("^.*__", "", group_id)
  ) %>%
  left_join(
    meta %>%
      select(donor, condition) %>%
      distinct(),
    by = "donor"
  )

peak_coldata
table(peak_coldata$rna_celltype, peak_coldata$condition)

In [ ]:
# ------------------------------------------------------------
# 8) Save donor-level ATAC matrices for downstream DA
#    (easy to use later in DESeq2/edgeR/limma if desired)
# ------------------------------------------------------------
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

# save the full grouped SE
saveRDS(se_peak_donor, file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_SE.rds"))

# save row metadata (peaks)
peak_ranges <- as.data.frame(rowRanges(se_peak_donor)) %>%
  rownames_to_column("peak_id")
write.table(
  peak_ranges,
  file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_peaks.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

# save grouped colData
write.table(
  peak_coldata,
  file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_coldata.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

# save one counts matrix per cell type
peak_mat <- assay(se_peak_donor)

for (ct in unique(peak_coldata$rna_celltype)) {
  cols_ct <- peak_coldata$group_id[peak_coldata$rna_celltype == ct]
  mat_ct <- peak_mat[, cols_ct, drop = FALSE]
  coldata_ct <- peak_coldata %>% filter(rna_celltype == ct)

  write.table(
    as.matrix(mat_ct),
    file.path(outdir_atac, paste0("PeakMatrix_", ct, "_counts.tsv")),
    sep = "\t", quote = FALSE, col.names = NA
  )

  write.table(
    coldata_ct,
    file.path(outdir_atac, paste0("PeakMatrix_", ct, "_coldata.tsv")),
    sep = "\t", quote = FALSE, row.names = FALSE
  )
}

In [ ]:
# ------------------------------------------------------------
# 9) Condition-specific coverage tracks by RNA-defined ATAC groups
# ------------------------------------------------------------
# build coverage groups for condition-specific visualization
archr_heme <- addGroupCoverages(
  ArchRProj = archr_heme,
  groupBy = "rna_celltype_condition",
  useLabels = TRUE,
  sampleLabels = "donor",
  minCells = 40,
  maxCells = 500,
  minReplicates = 2,
  maxReplicates = 5,
  force = TRUE
)

groups_to_plot <- c(
  "HSC_MPP__disomy", "HSC_MPP__Ts21",
  "B_primed_prog__disomy", "B_primed_prog__Ts21",
  "erythroid_prog__disomy", "erythroid_prog__Ts21",
  "erythroid_heme_prog__disomy", "erythroid_heme_prog__Ts21"
)

In [ ]:
# ------------------------------------------------------------
# 10) Compact browser tracks for main figure loci
# ------------------------------------------------------------

# only keep the 4 groups relevant to the main figure
groups_main <- c(
  "HSC_MPP__disomy", "HSC_MPP__Ts21",
  "B_primed_prog__disomy", "B_primed_prog__Ts21"
)

# set this to TRUE if you want the peak track;
# FALSE gives a cleaner/shorter figure panel
show_peak_track <- FALSE

plot_summary_use <- if (show_peak_track) {
  c("bulkTrack", "featureTrack", "geneTrack")
} else {
  c("bulkTrack", "geneTrack")
}

sizes_use <- if (show_peak_track) {
  c(8, 0.8, 1.8)   # bulk, peak, gene
} else {
  c(8, 1.5)        # bulk, gene
}

# helper function
make_compact_track <- function(gene_symbol,
                               up = 20000,
                               down = 20000,
                               base_size = 6,
                               ylim = c(0, 1),
                               facet_base_size = 6) {
  plotBrowserTrack(
    ArchRProj = archr_heme,
    geneSymbol = gene_symbol,
    groupBy = "rna_celltype_condition",
    useGroups = groups_main,
    upstream = up,
    downstream = down,
    plotSummary = plot_summary_use,
    sizes = sizes_use,
    ylim = c(0, 1),        # helps reduce clipping of tall peaks
    baseSize = base_size,
    facetbaseSize = facet_base_size
  )
}

# make compact plots for the main loci
p_ebf1 <- make_compact_track("EBF1", up = 20000, down = 20000)
p_pax5 <- make_compact_track("PAX5", up = 20000, down = 20000)
p_cd24 <- make_compact_track("CD24", up = 15000, down = 15000)
p_dntt <- make_compact_track("DNTT", up = 20000, down = 20000)

# save all together
plotPDF(
  p_ebf1, p_pax5, p_cd24, p_dntt,
  name = "ATAC_browserTracks_Blineage_compact.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 5,
  height = 4.8
)

In [ ]:
p_ebf1_compact <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "EBF1",
  groupBy = "rna_celltype_condition",
  useGroups = c(
    "HSC_MPP__disomy", "HSC_MPP__Ts21",
    "B_primed_prog__disomy", "B_primed_prog__Ts21"
  ),
  upstream = 20000,
  downstream = 20000,
  plotSummary = c("bulkTrack", "geneTrack"),
  sizes = c(8, 1.2),
  ylim = c(0, 1),
  baseSize = 5,
  facetbaseSize = 5,
  borderWidth = 0
)

plotPDF(
  p_ebf1_compact,
  name = "EBF1_browser_compact.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 6,
  height = 2
)

In [ ]:
# ------------------------------------------------------------
# Compact browser tracks for 4 main B-lineage loci
# ------------------------------------------------------------

groups_main <- c(
  "HSC_MPP__disomy", "HSC_MPP__Ts21",
  "B_primed_prog__disomy", "B_primed_prog__Ts21"
)

make_compact_track <- function(gene_symbol,
                               up = 10000,
                               down = 10000,
                               y_lim = c(0, 1),
                               base_size = 5,
                               facet_base_size = 5) {
  plotBrowserTrack(
    ArchRProj = archr_heme,
    geneSymbol = gene_symbol,
    groupBy = "rna_celltype_condition",
    useGroups = groups_main,
    upstream = up,
    downstream = down,
    plotSummary = c("bulkTrack", "geneTrack"),
    sizes = c(8, 1.2),
    ylim = y_lim,
    baseSize = base_size,
    facetbaseSize = facet_base_size
  )
}

# individual loci
p_ebf1 <- make_compact_track("EBF1", up = 10000, down = 10000)
p_pax5 <- make_compact_track("PAX5", up = 10000, down = 10000)
p_cd24 <- make_compact_track("CD24", up = 8000, down = 8000)
p_dntt <- make_compact_track("DNTT", up = 10000, down = 10000)

# save all together
plotPDF(
  p_ebf1, p_pax5, p_cd24, p_dntt,
  name = "ATAC_browserTracks_Blineage_compact4_3.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 4,
  height = 1.5
)

In [ ]:
# ------------------------------------------------------------
# 10) Browser tracks at key loci
# ------------------------------------------------------------
p_ebf1 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "EBF1",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_pax5 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "PAX5",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_cd24 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "CD24",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_dntt <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "DNTT",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_alas2 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "ALAS2",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_hbb <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "HBB",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_ifi6 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "IFI6",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

p_isg15 <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "ISG15",
  groupBy = "rna_celltype_condition",
  useGroups = groups_to_plot,
  upstream = 50000,
  downstream = 50000
)

plotPDF(
  p_ebf1, p_pax5, p_cd24, p_dntt,
  p_alas2, p_hbb, p_ifi6, p_isg15,
  name = "RNA_defined_ATAC_browser_tracks.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 12,
  height = 8
)

In [ ]:
# ------------------------------------------------------------
# 11) Exploratory per-cell DA inside ArchR (secondary, not the main donor-aware test)
# ------------------------------------------------------------
markers_hsc <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "HSC_MPP__Ts21",
  bgdGroups = "HSC_MPP__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

markers_bprimed <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "B_primed_prog__Ts21",
  bgdGroups = "B_primed_prog__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

saveRDS(markers_hsc, file.path(outdir_atac, "markers_HSC_MPP_Ts21_vs_disomy.rds"))
saveRDS(markers_bprimed, file.path(outdir_atac, "markers_B_primed_prog_Ts21_vs_disomy.rds"))

In [ ]:
# make sure these columns already exist in cellColData
meta <- as.data.frame(getCellColData(
  archr_heme,
  select = c("rna_celltype", "donor", "condition")
)) %>%
  rownames_to_column("cellNames") %>%
  mutate(
    rna_celltype_donor = paste(rna_celltype, donor, sep = "__"),
    rna_celltype_condition = paste(rna_celltype, condition, sep = "__")
  )

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$rna_celltype_donor,
  name = "rna_celltype_donor",
  cells = meta$cellNames,
  force = TRUE
)

archr_heme <- addCellColData(
  ArchRProj = archr_heme,
  data = meta$rna_celltype_condition,
  name = "rna_celltype_condition",
  cells = meta$cellNames,
  force = TRUE
)

# donor-level grouped PeakMatrix
se_peak_donor <- getGroupSE(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_donor",
  divideN = FALSE,
  scaleTo = NULL
)

# save the grouped donor-level ATAC object
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

saveRDS(se_peak_donor, file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_SE.rds"))

peak_coldata <- as.data.frame(colData(se_peak_donor)) %>%
  rownames_to_column("group_id") %>%
  mutate(
    rna_celltype = sub("__.*$", "", group_id),
    donor = sub("^.*__", "", group_id)
  ) %>%
  left_join(meta %>% select(donor, condition) %>% distinct(), by = "donor")

write.table(
  peak_coldata,
  file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_coldata.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

write.table(
  as.data.frame(rowRanges(se_peak_donor)) %>% rownames_to_column("peak_id"),
  file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_peaks.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

write.table(
  as.matrix(assay(se_peak_donor)),
  file.path(outdir_atac, "PeakMatrix_rnaCelltypeByDonor_counts.tsv"),
  sep = "\t", quote = FALSE, col.names = NA
)

In [ ]:
#2) Peak-level accessibility differences inside HSC_MPP and B_primed_prog
# exploratory per-cell DA inside ArchR (good for prioritization)
markers_hsc <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "HSC_MPP__Ts21",
  bgdGroups = "HSC_MPP__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

markers_b <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "PeakMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "B_primed_prog__Ts21",
  bgdGroups = "B_primed_prog__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

saveRDS(markers_hsc, file.path(outdir_atac, "markers_HSC_MPP_Ts21_vs_disomy.rds"))
saveRDS(markers_b, file.path(outdir_atac, "markers_B_primed_prog_Ts21_vs_disomy.rds"))

In [ ]:
#3) Motif enrichment: are Ts21-lost peaks depleted for B-cell TF motifs?
# add motif annotations once
archr_heme <- addMotifAnnotations(
  ArchRProj = archr_heme,
  motifSet = "cisbp",
  name = "Motif"
)

# enrich motifs in the disomy-higher peak sets
enrich_hsc_disomy <- peakAnnoEnrichment(
  seMarker = markers_hsc,
  ArchRProj = archr_heme,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC <= -0.5"   # Ts21 vs disomy: negative Log2FC => more open in disomy
)

enrich_b_disomy <- peakAnnoEnrichment(
  seMarker = markers_b,
  ArchRProj = archr_heme,
  peakAnnotation = "Motif",
  cutOff = "FDR <= 0.1 & Log2FC <= -0.5"
)

saveRDS(enrich_hsc_disomy, file.path(outdir_atac, "motif_enrichment_HSC_disomyHigher.rds"))
saveRDS(enrich_b_disomy, file.path(outdir_atac, "motif_enrichment_Bprimed_disomyHigher.rds"))

In [ ]:
saveRDS(enrich_hsc_disomy, "/path/to/nature_study/ATAC_donor_level/motif_enrichment_HSC_disomyHigher.rds")
saveRDS(enrich_b_disomy,   "/path/to/nature_study/ATAC_donor_level/motif_enrichment_Bprimed_disomyHigher.rds")

In [ ]:
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

# save the raw enrichment objects first
saveRDS(enrich_hsc_disomy, file.path(outdir_atac, "motif_enrichment_HSC_disomyHigher.rds"))
saveRDS(enrich_b_disomy,   file.path(outdir_atac, "motif_enrichment_Bprimed_disomyHigher.rds"))

# get the matrix behind the heatmap instead of drawing the ArchR object
mat_hsc <- plotEnrichHeatmap(
  enrich_hsc_disomy,
  n = 20,
  transpose = TRUE,
  returnMatrix = TRUE
)

mat_b <- plotEnrichHeatmap(
  enrich_b_disomy,
  n = 20,
  transpose = TRUE,
  returnMatrix = TRUE
)

# save matrices as TSV
write.table(
  mat_hsc,
  file.path(outdir_atac, "motif_enrichment_HSC_disomyHigher_matrix.tsv"),
  sep = "\t", quote = FALSE, col.names = NA
)

write.table(
  mat_b,
  file.path(outdir_atac, "motif_enrichment_Bprimed_disomyHigher_matrix.tsv"),
  sep = "\t", quote = FALSE, col.names = NA
)

In [ ]:
dim(mat_hsc)
dim(mat_b)

rownames(mat_hsc)
colnames(mat_hsc)

In [ ]:
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

save_enrich_plot <- function(mat, prefix, outdir) {
  mat <- as.matrix(mat)

  # save raw matrix
  write.table(
    mat,
    file.path(outdir, paste0(prefix, "_matrix.tsv")),
    sep = "\t", quote = FALSE, col.names = NA
  )

  nr <- nrow(mat)
  nc <- ncol(mat)

  message(prefix, ": ", nr, " rows x ", nc, " cols")

  pdf(file.path(outdir, paste0(prefix, "_plot.pdf")), width = 10, height = 8)

  if (nr >= 2 && nc >= 2) {
    heatmap(
      mat,
      Colv = NA,
      scale = "none",
      main = prefix
    )
  } else {
    vals <- as.numeric(mat)
    names(vals) <- rownames(mat)

    ord <- order(vals, decreasing = TRUE)
    vals <- vals[ord]

    barplot(
      vals,
      las = 2,
      cex.names = 0.7,
      main = prefix,
      ylab = "Norm. enrichment"
    )
  }

  dev.off()

  # also save a ranked table
  top_tbl <- data.frame(
    motif = rownames(mat),
    enrichment = as.numeric(mat[, 1])
  )
  top_tbl <- top_tbl[order(top_tbl$enrichment, decreasing = TRUE), ]

  write.table(
    top_tbl,
    file.path(outdir, paste0(prefix, "_ranked.tsv")),
    sep = "\t", quote = FALSE, row.names = FALSE
  )

  return(top_tbl)
}

top_hsc <- save_enrich_plot(
  mat_hsc,
  prefix = "HSC_MPP_disomyHigher_motif_enrichment",
  outdir = outdir_atac
)

top_b <- save_enrich_plot(
  mat_b,
  prefix = "B_primed_prog_disomyHigher_motif_enrichment",
  outdir = outdir_atac
)

head(top_hsc, 20)
head(top_b, 20)

In [ ]:
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

# get matrices in motif x group orientation
mat_hsc <- plotEnrichHeatmap(
  enrich_hsc_disomy,
  n = 20,
  transpose = FALSE,
  returnMatrix = TRUE
)

mat_b <- plotEnrichHeatmap(
  enrich_b_disomy,
  n = 20,
  transpose = FALSE,
  returnMatrix = TRUE
)

dim(mat_hsc)
dim(mat_b)

rownames(mat_hsc)
colnames(mat_hsc)

rownames(mat_b)
colnames(mat_b)





In [ ]:
make_motif_table <- function(mat) {
  mat <- as.matrix(mat)
  
  # if still 1 row x many cols, transpose it
  if (nrow(mat) == 1 && ncol(mat) > 1) {
    mat <- t(mat)
  }
  
  if (ncol(mat) == 1) {
    out <- data.frame(
      motif = rownames(mat),
      enrichment = as.numeric(mat[, 1]),
      stringsAsFactors = FALSE
    )
  } else {
    out <- data.frame(
      motif = rownames(mat),
      max_enrichment = apply(mat, 1, max, na.rm = TRUE),
      stringsAsFactors = FALSE
    )
  }
  
  out[order(out[[2]], decreasing = TRUE), , drop = FALSE]
}

top_hsc <- make_motif_table(mat_hsc)
top_b   <- make_motif_table(mat_b)

head(top_hsc, 20)
head(top_b, 20)

write.table(
  top_hsc,
  file.path(outdir_atac, "top_motifs_HSC_disomyHigher.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

write.table(
  top_b,
  file.path(outdir_atac, "top_motifs_Bprimed_disomyHigher.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

In [ ]:
pdf(file.path(outdir_atac, "top_motifs_barplots.pdf"), width = 10, height = 8)

barplot(
  head(top_hsc[[2]], 20),
  names.arg = head(top_hsc$motif, 20),
  las = 2,
  cex.names = 0.7,
  main = "HSC_MPP disomy-higher motif enrichment",
  ylab = colnames(top_hsc)[2]
)

barplot(
  head(top_b[[2]], 20),
  names.arg = head(top_b$motif, 20),
  las = 2,
  cex.names = 0.7,
  main = "B_primed_prog disomy-higher motif enrichment",
  ylab = colnames(top_b)[2]
)

dev.off()

In [ ]:
get_top_motifs <- function(enrich_obj, out_prefix, outdir) {
  df <- data.frame(
    motif = rownames(enrich_obj),
    mlog10Padj = assay(enrich_obj, "mlog10Padj")[, 1],
    stringsAsFactors = FALSE
  )

  df <- df[order(df$mlog10Padj, decreasing = TRUE), , drop = FALSE]

  write.table(
    df,
    file.path(outdir, paste0(out_prefix, "_raw_mlog10Padj.tsv")),
    sep = "\t", quote = FALSE, row.names = FALSE
  )

  df
}

outdir_atac <- "/path/to/nature_study/ATAC_donor_level"

top_hsc_raw <- get_top_motifs(
  enrich_hsc_disomy,
  "HSC_MPP_disomyHigher_motifs",
  outdir_atac
)

top_b_raw <- get_top_motifs(
  enrich_b_disomy,
  "B_primed_prog_disomyHigher_motifs",
  outdir_atac
)

head(top_hsc_raw, 20)
head(top_b_raw, 20)

In [ ]:
assayNames(enrich_hsc_disomy)
assayNames(enrich_b_disomy)

class(assay(enrich_hsc_disomy, "mlog10Padj"))
str(assay(enrich_hsc_disomy, "mlog10Padj"), max.level = 1)

In [ ]:
extract_motif_scores <- function(enrich_obj, assay_name = "mlog10Padj", column = 1) {
  x <- assay(enrich_obj, assay_name)

  # keep motif names from the enrichment object
  motifs <- rownames(enrich_obj)

  # handle common cases robustly
  if (is.matrix(x) || inherits(x, "Matrix")) {
    vals <- x[, column]
  } else if (is.data.frame(x)) {
    vals <- x[[column]]
  } else if (!is.null(dim(x))) {
    vals <- x[, column]
  } else if (is.list(x)) {
    vals <- x[[column]]
  } else {
    vals <- x
  }

  # flatten list-like / DataFrame-like columns safely
  vals <- unlist(vals, use.names = FALSE)

  # trim to motif count if needed
  if (length(vals) != length(motifs)) {
    stop(
      "Length mismatch: ", length(vals), " values but ",
      length(motifs), " motifs. Check assay structure with str()."
    )
  }

  df <- data.frame(
    motif = motifs,
    score = as.numeric(vals),
    stringsAsFactors = FALSE
  )

  df <- df[order(df$score, decreasing = TRUE), , drop = FALSE]
  df
}

top_hsc_raw <- extract_motif_scores(enrich_hsc_disomy, assay_name = "mlog10Padj", column = 1)
top_b_raw   <- extract_motif_scores(enrich_b_disomy,   assay_name = "mlog10Padj", column = 1)

head(top_hsc_raw, 20)
head(top_b_raw, 20)

In [ ]:
outdir_atac <- "/path/to/nature_study/ATAC_donor_level"
dir.create(outdir_atac, recursive = TRUE, showWarnings = FALSE)

write.table(
  top_hsc_raw,
  file.path(outdir_atac, "top_motifs_HSC_disomyHigher_raw.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

write.table(
  top_b_raw,
  file.path(outdir_atac, "top_motifs_Bprimed_disomyHigher_raw.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

plot_top_motifs <- function(df, title, outfile, n = 15) {
  df2 <- head(df, n)

  pdf(outfile, width = 10, height = 6)
  par(mar = c(10, 4, 4, 1))

  barplot(
    height = rev(df2$score),
    names.arg = rev(df2$motif),
    horiz = TRUE,
    las = 1,
    cex.names = 0.75,
    main = title,
    xlab = "-log10 adjusted P"
  )

  dev.off()
}

plot_top_motifs(
  top_hsc_raw,
  "HSC_MPP: motifs enriched in disomy-higher peaks",
  file.path(outdir_atac, "HSC_MPP_top_motifs_barplot.pdf"),
  n = 15
)

plot_top_motifs(
  top_b_raw,
  "B_primed_prog: motifs enriched in disomy-higher peaks",
  file.path(outdir_atac, "B_primed_prog_top_motifs_barplot.pdf"),
  n = 15
)

In [ ]:
#4) Motif activity per cell/group: chromVAR-style deviations
# background peaks for chromVAR-style deviation analysis
archr_heme <- addBgdPeaks(archr_heme, force = TRUE)

# motif deviations
archr_heme <- addDeviationsMatrix(
  ArchRProj = archr_heme,
  peakAnnotation = "Motif",
  matrixName = "MotifMatrix",
  force = TRUE
)

# variable motifs, optional quick QC
varDev <- getVarDeviations(archr_heme, name = "MotifMatrix", plot = TRUE)

# differential motif activity
motif_hsc <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "MotifMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "HSC_MPP__Ts21",
  bgdGroups = "HSC_MPP__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

motif_b <- getMarkerFeatures(
  ArchRProj = archr_heme,
  useMatrix = "MotifMatrix",
  groupBy = "rna_celltype_condition",
  useGroups = "B_primed_prog__Ts21",
  bgdGroups = "B_primed_prog__disomy",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

saveRDS(motif_hsc, file.path(outdir_atac, "motifMarkers_HSC_Ts21_vs_disomy.rds"))
saveRDS(motif_b, file.path(outdir_atac, "motifMarkers_Bprimed_Ts21_vs_disomy.rds"))

In [ ]:
#5) Browser tracks at the exact genes you care about
groups_to_plot <- c(
  "HSC_MPP__disomy", "HSC_MPP__Ts21",
  "B_primed_prog__disomy", "B_primed_prog__Ts21",
  "erythroid_prog__disomy", "erythroid_prog__Ts21",
  "erythroid_heme_prog__disomy", "erythroid_heme_prog__Ts21"
)

# make sure group coverages exist for condition-specific groups
archr_heme <- addGroupCoverages(
  ArchRProj = archr_heme,
  groupBy = "rna_celltype_condition",
  useLabels = TRUE,
  sampleLabels = "donor",
  minCells = 40,
  maxCells = 500,
  minReplicates = 2,
  maxReplicates = 5,
  force = TRUE
)

p_ebf1 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "EBF1", upstream = 50000, downstream = 50000)
p_pax5 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "PAX5", upstream = 50000, downstream = 50000)
p_cd24 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "CD24", upstream = 50000, downstream = 50000)
p_dntt <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "DNTT", upstream = 50000, downstream = 50000)

p_gata1 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "GATA1", upstream = 50000, downstream = 50000)
p_alas2 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "ALAS2", upstream = 50000, downstream = 50000)
p_hbb   <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "HBB", upstream = 50000, downstream = 50000)

p_ifi6  <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "IFI6", upstream = 50000, downstream = 50000)
p_isg15 <- plotBrowserTrack(archr_heme, groupBy = "rna_celltype_condition", useGroups = groups_to_plot, geneSymbol = "ISG15", upstream = 50000, downstream = 50000)

plotPDF(
  p_ebf1, p_pax5, p_cd24, p_dntt,
  p_gata1, p_alas2, p_hbb, p_ifi6, p_isg15,
  name = "ATAC_browserTracks_Blineage_erythroid_IFN.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE
)

In [ ]:
archr_heme <- addIterativeLSI(
  ArchRProj = archr_heme,
  useMatrix = "TileMatrix",
  name = "IterativeLSI",
  iterations = 2,
  clusterParams = list(resolution = c(0.2), sampleCells = 10000, n.start = 10),
  varFeatures = 25000,
  dimsToUse = 1:30,
  force = TRUE
)

In [ ]:
names(archr_heme@reducedDims)

In [ ]:
# rna =  Seurat object  loaded
# archr_heme = ArchR project already loaded

rna <- readRDS("/path/to/nature_study/RNA_final_results/rna_full_final.rds")

archr_heme <- addGeneIntegrationMatrix(
  ArchRProj = archr_heme,
  useMatrix = "GeneScoreMatrix",
  matrixName = "GeneIntegrationMatrix",
  reducedDims = "IterativeLSI",
  seRNA = rna,
  addToArrow = TRUE,
  groupRNA = "celltype_rna",   # change if RNA annotation column has another name
  nameCell = "predictedCell",
  nameGroup = "predictedGroup",
  nameScore = "predictedScore",
  dimsToUse = 1:30,
  nGenes = 2000,
  force = TRUE
)

getAvailableMatrices(archr_heme)

In [ ]:
# ------------------------------------------------------------
# 1) choose cells
# ------------------------------------------------------------
celltype_col <- c("celltype_rna", "rna_celltype", "celltype")[
  c("celltype_rna", "rna_celltype", "celltype") %in% colnames(getCellColData(archr_heme))
][1]

if (length(celltype_col) == 0) {
  stop("Could not find cell-type annotation column in ArchR project.")
}

meta <- as.data.frame(getCellColData(archr_heme))
states_keep <- c("HSC_MPP", "B_primed_prog", "erythroid_prog", "erythroid_heme_prog")
cells_keep <- rownames(meta)[meta[[celltype_col]] %in% states_keep]

cat("Cells retained for peak-to-gene links:\n")
print(table(meta[cells_keep, celltype_col], useNA = "ifany"))

# choose reducedDims name automatically
reduced_use <- if ("IterativeLSI" %in% names(archr_heme@reducedDims)) {
  "IterativeLSI"
} else {
  names(archr_heme@reducedDims)[1]
}

cat("Using reducedDims:", reduced_use, "\n")

In [ ]:
# ------------------------------------------------------------
# 2) add peak-to-gene links
# ------------------------------------------------------------
archr_heme <- addPeak2GeneLinks(
  ArchRProj = archr_heme,
  reducedDims = reduced_use,
  useMatrix = "GeneIntegrationMatrix",
  dimsToUse = 1:30,
  cellsToUse = cells_keep,
  k = 100,
  knnIteration = 500,
  overlapCutoff = 0.8,
  maxDist = 250000,
  corCutOff = 0.75,
  predictionCutoff = 0.4
)

In [ ]:
# ------------------------------------------------------------
# 3) retrieve links as a table
# ------------------------------------------------------------
p2g_df <- getPeak2GeneLinks(
  ArchRProj = archr_heme,
  corCutOff = 0.45,
  FDRCutOff = 1e-04,
  resolution = 1,
  returnLoops = FALSE
)

head(p2g_df)
dim(p2g_df)

In [ ]:
peak_set <- metadata(p2g_df)$peakSet
gene_set <- metadata(p2g_df)$geneSet

gene_meta <- as.data.frame(mcols(gene_set))
gene_name_col <- c("symbol", "name", "gene_name", "SYMBOL")[
  c("symbol", "name", "gene_name", "SYMBOL") %in% colnames(gene_meta)
][1]

gene_names <- if (!is.na(gene_name_col)) {
  gene_meta[[gene_name_col]][p2g_df$idxRNA]
} else if (!is.null(names(gene_set))) {
  names(gene_set)[p2g_df$idxRNA]
} else {
  as.character(p2g_df$idxRNA)
}

p2g_tab <- tibble(
  peak_chr   = as.character(seqnames(peak_set[p2g_df$idxATAC])),
  peak_start = start(peak_set[p2g_df$idxATAC]),
  peak_end   = end(peak_set[p2g_df$idxATAC]),
  gene       = gene_names,
  gene_chr   = as.character(seqnames(gene_set[p2g_df$idxRNA])),
  gene_start = start(gene_set[p2g_df$idxRNA]),
  gene_end   = end(gene_set[p2g_df$idxRNA]),
  correlation = p2g_df$Correlation,
  FDR         = p2g_df$FDR,
  varQ_atac   = p2g_df$VarQATAC,
  varQ_rna    = p2g_df$VarQRNA
) %>%
  arrange(FDR, desc(correlation))

genes_focus <- c("EBF1", "PAX5", "CD24", "DNTT", "BLNK", "IRF8", "IGLL1", "CD79A", "CD79B", "TCF3")

p2g_focus <- p2g_tab %>%
  filter(gene %in% genes_focus) %>%
  arrange(gene, FDR, desc(correlation))

outdir_p2g <- "/path/to/nature_study/ATAC_donor_level/peak2gene_links"
dir.create(outdir_p2g, recursive = TRUE, showWarnings = FALSE)

write.table(
  p2g_tab,
  file = file.path(outdir_p2g, "restricted_progenitors_peak2gene_links.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

write.table(
  p2g_focus,
  file = file.path(outdir_p2g, "restricted_progenitors_peak2gene_links_focusGenes.tsv"),
  sep = "\t", quote = FALSE, row.names = FALSE
)

p2g_focus %>% group_by(gene) %>% slice_head(n = 10) %>% ungroup()

In [ ]:
p2g_loops <- getPeak2GeneLinks(
  ArchRProj = archr_heme,
  corCutOff = 0.45,
  FDRCutOff = 1e-04,
  resolution = 1,
  returnLoops = TRUE
)

saveRDS(
  p2g_loops,
  file.path(outdir_p2g, "restricted_progenitors_peak2gene_loops.rds")
)

In [ ]:
# make pretty labels for browser-track plotting
plot_group <- as.character(archr_heme$rna_celltype_condition)

plot_group[plot_group == "HSC_MPP__disomy"] <- "HSC MPP D"
plot_group[plot_group == "HSC_MPP__Ts21"]   <- "HSC MPP Ts21"
plot_group[plot_group == "B_primed_prog__disomy"] <- "B Prog D"
plot_group[plot_group == "B_primed_prog__Ts21"]   <- "B Prog Ts21"

archr_heme$rna_celltype_condition_plot <- plot_group

table(archr_heme$rna_celltype_condition_plot, useNA = "ifany")

In [ ]:
archr_heme <- addGroupCoverages(
  ArchRProj = archr_heme,
  groupBy = "rna_celltype_condition_plot",
  useLabels = TRUE,
  sampleLabels = "donor",
  minCells = 40,
  maxCells = 500,
  minReplicates = 2,
  maxReplicates = 5,
  force = TRUE
)

In [ ]:
p2g_loops <- getPeak2GeneLinks(
  ArchRProj = archr_heme,
  corCutOff = 0.45,
  FDRCutOff = 1e-04,
  resolution = 1,
  returnLoops = TRUE
)

p_ebf1_p2g <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "EBF1",
  groupBy = "rna_celltype_condition_plot",
  useGroups = c("HSC MPP D", "HSC MPP Ts21", "B Prog D", "B Prog Ts21"),
  loops = p2g_loops,
  plotSummary = c("bulkTrack", "loopTrack", "geneTrack"),
  sizes = c(8, 2, 1.2),
  upstream = 10000,
  downstream = 10000,
  ylim = c(0, 1),
  baseSize = 5,
  facetbaseSize = 5
)

plotPDF(
  p_ebf1_p2g,
  name = "EBF1_browser_with_peak2gene_loops.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 5,
  height = 2.5
)

In [ ]:
p_pax5_p2g <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "PAX5",
  groupBy = "rna_celltype_condition_plot",
  useGroups = c("HSC MPP D", "HSC MPP Ts21", "B Prog D", "B Prog Ts21"),
  loops = p2g_loops,
  plotSummary = c("bulkTrack", "loopTrack", "geneTrack"),
  sizes = c(8, 2, 1.2),
  upstream = 10000,
  downstream = 10000,
  ylim = c(0, 1),
  baseSize = 5,
  facetbaseSize = 5
)

plotPDF(
  p_pax5_p2g,
  name = "PAX5_browser_with_peak2gene_loops.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 5,
  height = 2.5
)

In [ ]:
p_cd24_p2g <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "CD24",
  groupBy = "rna_celltype_condition_plot",
  useGroups = c("HSC MPP D", "HSC MPP Ts21", "B Prog D", "B Prog Ts21"),
  loops = p2g_loops,
  plotSummary = c("bulkTrack", "loopTrack", "geneTrack"),
  sizes = c(8, 2, 1.2),
  upstream = 8000,
  downstream = 8000,
  ylim = c(0, 1),
  baseSize = 5,
  facetbaseSize = 5
)



plotPDF(
  p_cd24_p2g,
  name = "CD24_browser_with_peak2gene_loops.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 5,
  height = 2.5
)

In [ ]:
p_dntt_p2g <- plotBrowserTrack(
  ArchRProj = archr_heme,
  geneSymbol = "DNTT",
  groupBy = "rna_celltype_condition_plot",
  useGroups = c("HSC MPP D", "HSC MPP Ts21", "B Prog D", "B Prog Ts21"),
  loops = p2g_loops,
  plotSummary = c("bulkTrack", "loopTrack", "geneTrack"),
  sizes = c(8, 2, 1.2),
  upstream = 10000,
  downstream = 10000,
  ylim = c(0, 1),
  baseSize = 5,
  facetbaseSize = 5
)



plotPDF(
  p_dntt_p2g,
  name = "DNTT_browser_with_peak2gene_loops.pdf",
  ArchRProj = archr_heme,
  addDOC = FALSE,
  width = 5,
  height = 2.5
)